# 09 — Análise Comparativa — Dataset Completo

Compara as 6 variantes no dataset completo (~20.639 tweets) e contrasta com os resultados do sample (500 tweets).

- Métrica principal: **F1-macro**
- Destaque para classes raras (`racism`, `misogyny`, `xenophobia`) com support adequado na base completa

In [ ]:
import json
from pathlib import Path

import polars as pl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from sklearn.metrics import (
    classification_report, f1_score, precision_score, recall_score, confusion_matrix
)

CATEGORIAS = ["homophobia", "insult", "misogyny", "not_toxic", "obscene", "racism", "xenophobia"]

EXPERIMENTOS_CONFIG = {
    "zs_v1_base":         ("../results/full/zero_shot_v1_base.csv",        "ZS-v1 Base",         "zero_shot"),
    "zs_v2_descriptions": ("../results/full/zero_shot_v2_descriptions.csv", "ZS-v2 Descriptions", "zero_shot"),
    "zs_v3_no_antibias":  ("../results/full/zero_shot_v3_no_antibias.csv",  "ZS-v3 No-Antibias",  "zero_shot"),
    "fs_v1_1ex":          ("../results/full/few_shot_v1_1ex.csv",           "FS-v1 1-Example",    "few_shot"),
    "fs_v2_2ex_antibias": ("../results/full/few_shot_v2_2ex_antibias.csv",  "FS-v2 2ex+Antibias", "few_shot"),
    "fs_v3_2ex":          ("../results/full/few_shot_v3_2ex.csv",           "FS-v3 2-Examples",   "few_shot"),
}

CORES = {"zero_shot": "#4C72B0", "few_shot": "#DD8452"}

experimentos = {}
for chave, (path, label, grupo) in EXPERIMENTOS_CONFIG.items():
    df = pl.read_csv(path).with_columns(pl.col("predicao").str.to_lowercase())
    unknown = (df["predicao"] == "unknown").sum()
    print(f"{label:<22}: {len(df)} tweets | unknown={unknown}")
    experimentos[chave] = df

## F1-macro por estratégia

In [ ]:
metricas = {}

for chave, df in experimentos.items():
    _, label, grupo = EXPERIMENTOS_CONFIG[chave]
    y_true = df["label"].to_list()
    y_pred = df["predicao"].to_list()
    metricas[chave] = {
        "label": label,
        "grupo": grupo,
        "f1_macro":        round(f1_score(y_true, y_pred, average="macro",    zero_division=0), 4),
        "f1_weighted":     round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "precision_macro": round(precision_score(y_true, y_pred, average="macro", zero_division=0), 4),
        "recall_macro":    round(recall_score(y_true, y_pred,    average="macro", zero_division=0), 4),
        "per_class": {},
    }
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    for cat in CATEGORIAS:
        if cat in report:
            metricas[chave]["per_class"][cat] = {
                "f1":        round(report[cat]["f1-score"],  4),
                "precision": round(report[cat]["precision"], 4),
                "recall":    round(report[cat]["recall"],    4),
                "support":   int(report[cat]["support"]),
            }

print(f"{'Estratégia':<22} {'F1-macro':>10} {'F1-weighted':>12} {'Precision':>10} {'Recall':>8}")
print("-" * 66)
for chave, m in metricas.items():
    print(f"{m['label']:<22} {m['f1_macro']:>10.4f} {m['f1_weighted']:>12.4f} {m['precision_macro']:>10.4f} {m['recall_macro']:>8.4f}")

## Ranking por F1-macro

In [ ]:
ranking = sorted(metricas.items(), key=lambda x: x[1]["f1_macro"], reverse=True)

fig, ax = plt.subplots(figsize=(9, 4))
labels_r = [m["label"] for _, m in ranking]
f1s_r    = [m["f1_macro"] for _, m in ranking]
cores_r  = [CORES[m["grupo"]] for _, m in ranking]

bars = ax.barh(labels_r, f1s_r, color=cores_r, alpha=0.85)
for bar, val in zip(bars, f1s_r):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2, f"{val:.4f}", va="center", fontsize=9)

ax.set_xlabel("F1-macro")
ax.set_title("Ranking — F1-macro por variante (dataset completo)")
ax.set_xlim(0, max(f1s_r) * 1.15)
ax.grid(axis="x", alpha=0.3)
legend = [
    mpatches.Patch(color=CORES["zero_shot"], label="Zero-Shot"),
    mpatches.Patch(color=CORES["few_shot"],  label="Few-Shot"),
]
ax.legend(handles=legend, loc="lower right")
plt.tight_layout()
plt.savefig("../results/full/ranking_f1_macro.png", dpi=150)
plt.show()

melhor_chave, melhor_m = ranking[0]
print(f"\nMelhor variante: {melhor_m['label']} — F1-macro={melhor_m['f1_macro']:.4f}")

## F1 por categoria — todas as variantes

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
n = len(metricas)
x = np.arange(len(CATEGORIAS))
width = 0.13
paleta_zs = ["#1f4e79", "#2e75b6", "#9dc3e6"]
paleta_fs = ["#833c00", "#c55a11", "#f4b183"]
palettas  = paleta_zs + paleta_fs

for idx, (chave, m) in enumerate(metricas.items()):
    f1s = [m["per_class"].get(cat, {}).get("f1", 0) for cat in CATEGORIAS]
    offset = (idx - n / 2 + 0.5) * width
    ax.bar(x + offset, f1s, width, label=m["label"], color=palettas[idx], alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(CATEGORIAS, rotation=20, ha="right")
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1.05)
ax.set_title("F1 por categoria — dataset completo")
ax.legend(fontsize=8, ncol=2)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("../results/full/f1_por_categoria.png", dpi=150)
plt.show()

## Matrizes de confusão — todas as variantes

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (chave, df) in zip(axes.flatten(), experimentos.items()):
    m = metricas[chave]
    y_true = df["label"].to_list()
    y_pred = df["predicao"].to_list()
    cm = confusion_matrix(y_true, y_pred, labels=CATEGORIAS)
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=CATEGORIAS, yticklabels=CATEGORIAS,
                cmap="Blues", ax=ax, cbar=False)
    ax.set_xlabel("Predito", fontsize=8)
    ax.set_ylabel("Real", fontsize=8)
    ax.set_title(f"{m['label']}\nF1-macro={m['f1_macro']:.4f}", fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0,  labelsize=7)
plt.tight_layout()
plt.savefig("../results/full/confusion_matrices.png", dpi=150)
plt.show()

## Comparação: Sample (500) vs Full (~20.639)

In [ ]:
sample_summary = json.load(open("../results/metrics_summary.json", encoding="utf-8"))

CHAVE_MAP = {
    "zs_v1_base": "zs_v1_base",
    "zs_v2_descriptions": "zs_v2_descriptions",
    "zs_v3_no_antibias": "zs_v3_no_antibias",
    "fs_v1_1ex": "fs_v1_1ex",
    "fs_v2_2ex_antibias": "fs_v2_2ex_antibias",
    "fs_v3_2ex": "fs_v3_2ex",
}

print(f"{'Variante':<22} {'Sample F1':>10} {'Full F1':>10} {'Δ':>8}")
print("-" * 54)
for chave, m_full in metricas.items():
    m_sample = sample_summary.get(chave, {})
    f1_sample = m_sample.get("f1_macro", float("nan"))
    f1_full   = m_full["f1_macro"]
    delta = f1_full - f1_sample
    sign = "+" if delta >= 0 else ""
    print(f"{m_full['label']:<22} {f1_sample:>10.4f} {f1_full:>10.4f} {sign}{delta:>7.4f}")

## Classes raras — análise detalhada

In [ ]:
RARAS = ["racism", "misogyny", "xenophobia", "homophobia"]

print(f"{'Variante':<22}", end="")
for cat in RARAS:
    print(f" {cat:>12}", end="")
print()
print("-" * (22 + 13 * len(RARAS)))

for chave, m in metricas.items():
    print(f"{m['label']:<22}", end="")
    for cat in RARAS:
        f1 = m["per_class"].get(cat, {}).get("f1", 0.0)
        sup = m["per_class"].get(cat, {}).get("support", 0)
        print(f" {f1:>6.3f}({sup:>4})", end="")
    print()

## Salvar resultados

In [ ]:
with open("../results/full/metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)
print("Salvo: results/full/metrics_summary.json")

rows = []
for chave, m in metricas.items():
    for cat in CATEGORIAS:
        pc = m["per_class"].get(cat, {})
        rows.append({
            "estrategia": chave, "label": m["label"], "grupo": m["grupo"],
            "categoria": cat,
            "f1": pc.get("f1", 0.0), "precision": pc.get("precision", 0.0),
            "recall": pc.get("recall", 0.0), "support": pc.get("support", 0),
            "f1_macro": m["f1_macro"], "f1_weighted": m["f1_weighted"],
        })
pl.DataFrame(rows).write_csv("../results/full/metrics_per_strategy.csv")
print("Salvo: results/full/metrics_per_strategy.csv")

print("\nRESUMO FINAL — ranking por F1-macro (dataset completo)")
print("=" * 55)
for chave, m in sorted(metricas.items(), key=lambda x: x[1]["f1_macro"], reverse=True):
    print(f"{m['label']:<22}  F1-macro={m['f1_macro']:.4f}  F1-weighted={m['f1_weighted']:.4f}")